# Unsloth training on Colab

Minimal setup: clone repo → install deps → run GRPO training with Unsloth (Qwen3-4B, 4-bit + LoRA).

**Runtime**: Enable a GPU (e.g. T4) in Colab: Runtime → Change runtime type → GPU.

In [ ]:
# 1. Clone repo (set branch/tag if needed)
REPO_URL = "https://github.com/mhtruong1031/OpenENV-Hackathon.git"  # or your fork
REPO_DIR = "OpenENV-Hackathon"

!git clone --depth 1 {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
# 2. Install requirements: project + train extras + Unsloth (no-deps to keep trl>=0.29)
!pip install -q -e ".[train]"
!pip install -q unsloth unsloth_zoo --no-deps

# Optional: reward backends
!pip install -q sentence-transformers gseapy 2>/dev/null || true

In [ ]:
# 3. Unsloth must be imported before trl/transformers/peft
import unsloth  # noqa: F401
import torch
from pathlib import Path

from training_unsloth import make_training_args, run_training

print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
Path("artifacts").mkdir(exist_ok=True)

In [ ]:
# 4. Training config (small run for Colab T4)
args = make_training_args(
    model_id="Qwen/Qwen3-4B-Base",
    output_dir="artifacts/grpo-unsloth-qwen3-4b",
    dataset_episodes=16,
    rollout_steps=10,
    collection_policy="heuristic",
    reward_backend="local",
    domain_randomise=True,
    num_generations=4,
    max_completion_length=160,
    max_prompt_length=1280,
    max_seq_length=2048,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_train_epochs=1.0,
    logging_steps=1,
    save_steps=25,
    trust_remote_code=True,
    dry_run=False,
    seed=42,
)
args

In [ ]:
# 5. Run training
result = run_training(args)
print("Plots:", result["plot_paths"])

In [ ]:
# 6. (Optional) Show loss curves
from IPython.display import Image, display
for name, path in result["plot_paths"].items():
    display(Image(filename=path))